Los repositorios y datasets oficiales para empezar a experimentar son:Para Grafos y Comunidades:TwiBot-22 / TwiBot-20: Archivo del repositorio en GitHub (BunsenFeng/TwiBot-20) y ficha técnica en Botometer Repository.Cresci-2017: Disponible en Kaggle (hashemalsalmi/cresci-2017) y el subset de pruebas en GitHub (ashu273k/TrustNet).  Para Texto y Similitud Semántica:MGTBench: Disponible en GitHub (MBZUAI-NLP/MGTBench).

In [ ]:
!pip install pandas networkx python-louvain sentence-transformers scikit-learn

In [ ]:
!pip install pyvis #iat_std_dev

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 77.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import networkx as nx
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import IsolationForest
import numpy as np
import matplotlib.pyplot as plt
#from pyvis.network import Network

import fastapi
from fastapi import FastAPI

from community import community_louvain
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:


# Leer el dataset de tweets
try:
    df = pd.read_csv('dataset_tweets.csv')
except FileNotFoundError:
    df = pd.DataFrame({
        'user_id': ['bot_1', 'bot_1', 'bot_2', 'bot_2', 'user_A', 'user_A', 'user_B'],
        'mentioned_user': ['bot_2', 'bot_2', 'bot_1', 'bot_1', 'user_B', 'user_C', 'user_A'],
        'text': [
            '¡Atención! Campaña importante sobre inversión cripto #CryptoBot',
            '¡Atención! Campaña importante sobre inversión cripto #CryptoBot',
            '¡Atención! Campaña importante sobre inversión cripto #CryptoBot',
            '¡Atención! Campaña importante sobre inversión cripto #CryptoBot',
            'Hoy fui al parque y el clima estuvo genial.',
            '¿Alguien recomienda un buen libro de ciencia ficción?',
            'Coincido totalmente con tu posteo anterior.'
        ],
        'created_at': [
            '2026-09-01 10:00:00', '2026-09-01 10:00:02',
            '2026-09-01 10:00:01', '2026-09-01 10:00:03',
            '2026-09-01 12:30:15', '2026-09-01 18:45:00',
            '2026-09-01 13:10:05'
        ]
    })

In [ ]:
df.head()

,user_id,mentioned_user,text,created_at
0,bot_1,bot_2,¡Atención! Campaña importante sobre inversión ...,2026-09-01 10:00:00
1,bot_1,bot_2,¡Atención! Campaña importante sobre inversión ...,2026-09-01 10:00:02
2,bot_2,bot_1,¡Atención! Campaña importante sobre inversión ...,2026-09-01 10:00:01
3,bot_2,bot_1,¡Atención! Campaña importante sobre inversión ...,2026-09-01 10:00:03
4,user_A,user_B,Hoy fui al parque y el clima estuvo genial.,2026-09-01 12:30:15


In [ ]:
import community as community_louvain
from sklearn.metrics.pairwise import cosine_similarity
df['timestamp'] = pd.to_datetime(df['created_at']).astype('int64') // 10**9

# ==========================================
# 2. CONSTRUCCIÓN DEL GRAFO
# ==========================================
G = nx.Graph()

for _, row in df.iterrows():
    u_src = str(row['user_id'])
    u_dst = str(row['mentioned_user']) if pd.notna(row.get('mentioned_user')) else None

    G.add_node(u_src)
    if u_dst and u_dst != 'nan' and u_dst != u_src:
        G.add_edge(u_src, u_dst)

partition = community_louvain.best_partition(G)

# ==========================================
# 3. EMBEDDINGS (MANEJO DE USUARIOS PASIVOS)
# ==========================================
print("Cargando modelo de Embeddings...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

text_embeddings = model.encode(df['text'].tolist(), show_progress_bar=False)
df['embedding'] = list(text_embeddings)

user_embeddings = {}
user_timestamps = {}

# Ingesta para usuarios activos (que publicaron texto)
for user, group in df.groupby('user_id'):
    user_str = str(user)
    user_embeddings[user_str] = np.mean(np.array(group['embedding'].tolist()), axis=0)
    user_timestamps[user_str] = group['timestamp'].sort_values().tolist()

# Asignar valores por defecto a usuarios mensionados que NO publicaron nada (como user_C)
embedding_dim = text_embeddings.shape[1]
all_users = list(partition.keys())

for u in all_users:
    if u not in user_embeddings:
        # Vector cero para usuarios sin historial de texto
        user_embeddings[u] = np.zeros(embedding_dim)
        user_timestamps[u] = []

user_matrix = np.array([user_embeddings[u] for u in all_users])
user_to_idx = {user: idx for idx, user in enumerate(all_users)}

sim_matrix = cosine_similarity(user_matrix)

# ==========================================
# 4. MOTOR MULTI-CAPA Y REPORTE
# ==========================================
results = []

for comm_id in set(partition.values()):
    users_in_comm = [user for user, c in partition.items() if c == comm_id]

    if len(users_in_comm) > 1:
        indices = [user_to_idx[u] for u in users_in_comm]
        sub_matrix = sim_matrix[np.ix_(indices, indices)]

        # Excluir la diagonal y evitar ceros de usuarios pasivos en el promedio
        mask = ~np.eye(sub_matrix.shape[0], dtype=bool)
        avg_sim = sub_matrix[mask].mean() if mask.any() else 0.0

        sub_graph = G.subgraph(users_in_comm)
        density = nx.density(sub_graph)

        comm_iat_std = []
        for u in users_in_comm:
            ts = user_timestamps[u]
            if len(ts) > 1:
                iats = np.diff(ts)
                comm_iat_std.append(np.std(iats))
            else:
                comm_iat_std.append(300.0)

        avg_iat_std = np.mean(comm_iat_std)

        risk_score = 0
        if density >= 0.5: risk_score += 35
        if avg_sim >= 0.7: risk_score += 40
        if avg_iat_std <= 2.0: risk_score += 25

        sospecha = 'ALTA' if risk_score >= 60 else ('MEDIA' if risk_score >= 30 else 'BAJA')

        results.append({
            'Comunidad': comm_id,
            'Usuarios': len(users_in_comm),
            'Densidad Grafo': round(density, 3),
            'Simil. Semántica': round(avg_sim, 4),
            'Varianza Temp. (s)': round(avg_iat_std, 2),
            'Risk Score': risk_score,
            'Sospecha': sospecha
        })

report_df = pd.DataFrame(results)
print("\n--- INFORME DE DETECCIÓN MULTI-CAPA SOBRE DATASET REAL ---")
print(report_df.to_string(index=False))

Cargando modelo de Embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- INFORME DE DETECCIÓN MULTI-CAPA SOBRE DATASET REAL ---
 Comunidad  Usuarios  Densidad Grafo  Simil. Semántica  Varianza Temp. (s)  Risk Score Sospecha
         0         3           0.667            0.0993               200.0          35    MEDIA
         1         2           1.000            1.0000                 0.0         100     ALTA


In [ ]:

# ==========================================
# 1. DATOS DE ENTRADA
# ==========================================
try:
    df = pd.read_csv('dataset_tweets.csv')
except FileNotFoundError:
    df = pd.DataFrame({
        'user_id': ['bot_1', 'bot_1', 'bot_2', 'bot_2', 'user_A', 'user_A', 'user_B'],
        'mentioned_user': ['bot_2', 'bot_2', 'bot_1', 'bot_1', 'user_B', 'user_C', 'user_A'],
        'text': [
            '¡Atención! Campaña importante sobre inversión cripto #CryptoBot',
            '¡Atención! Campaña importante sobre inversión cripto #CryptoBot',
            '¡Atención! Campaña importante sobre inversión cripto #CryptoBot',
            '¡Atención! Campaña importante sobre inversión cripto #CryptoBot',
            'Hoy fui al parque y el clima estuvo genial.',
            '¿Alguien recomienda un buen libro de ciencia ficción?',
            'Coincido totalmente con tu posteo anterior.'
        ],
        'created_at': [
            '2026-09-01 10:00:00', '2026-09-01 10:00:02',
            '2026-09-01 10:00:01', '2026-09-01 10:00:03',
            '2026-09-01 12:30:15', '2026-09-01 18:45:00',
            '2026-09-01 13:10:05'
        ]
    })

df['timestamp'] = pd.to_datetime(df['created_at']).astype('int64') // 10**9

# ==========================================
# 2. GRAFO Y EMBEDDINGS
# ==========================================
G = nx.Graph()
for _, row in df.iterrows():
    u_src = str(row['user_id'])
    u_dst = str(row['mentioned_user']) if pd.notna(row.get('mentioned_user')) else None
    G.add_node(u_src)
    if u_dst and u_dst != 'nan' and u_dst != u_src:
        G.add_edge(u_src, u_dst)

partition = community_louvain.best_partition(G)

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
text_embeddings = model.encode(df['text'].tolist(), show_progress_bar=False)
df['embedding'] = list(text_embeddings)

user_embeddings = {}
user_timestamps = {}
for user, group in df.groupby('user_id'):
    user_str = str(user)
    user_embeddings[user_str] = np.mean(np.array(group['embedding'].tolist()), axis=0)
    user_timestamps[user_str] = group['timestamp'].sort_values().tolist()

embedding_dim = text_embeddings.shape[1]
all_users = list(partition.keys())
for u in all_users:
    if u not in user_embeddings:
        user_embeddings[u] = np.zeros(embedding_dim)
        user_timestamps[u] = []

user_matrix = np.array([user_embeddings[u] for u in all_users])
user_to_idx = {user: idx for idx, user in enumerate(all_users)}
sim_matrix = cosine_similarity(user_matrix)

# ==========================================
# 3. EXTRACCIÓN DE VECTORES DE CARACTERÍSTICAS
# ==========================================
features_list = []
community_metadata = []

for comm_id in set(partition.values()):
    users_in_comm = [user for user, c in partition.items() if c == comm_id]

    indices = [user_to_idx[u] for u in users_in_comm]
    sub_matrix = sim_matrix[np.ix_(indices, indices)]
    mask = ~np.eye(sub_matrix.shape[0], dtype=bool)
    avg_sim = sub_matrix[mask].mean() if mask.any() else 0.0

    sub_graph = G.subgraph(users_in_comm)
    density = nx.density(sub_graph)

    comm_iat_std = []
    for u in users_in_comm:
        ts = user_timestamps[u]
        if len(ts) > 1:
            iats = np.diff(ts)
            comm_iat_std.append(np.std(iats))
        else:
            comm_iat_std.append(300.0)
    avg_iat_std = np.mean(comm_iat_std)

    features_list.append([density, avg_sim, avg_iat_std])
    community_metadata.append({
        'comm_id': comm_id,
        'users': users_in_comm,
        'density': density,
        'avg_sim': avg_sim,
        'avg_iat_std': avg_iat_std
    })

X_real = np.array(features_list)

# ==========================================
# 4. MOTOR HÍBRIDO ML + REGLAS DE SEGURIDAD
# ==========================================
# Inyectamos una matriz de referencia humana artificial para estabilizar el modelo en muestras pequeñas
baseline_human_data = np.array([
    [0.10, 0.01, 250.0],
    [0.15, -0.02, 180.0],
    [0.08, 0.05, 320.0],
    [0.20, 0.00, 150.0]
])

X_train = np.vstack([baseline_human_data, X_real])

iso_forest = IsolationForest(contamination=0.25, random_state=42)
iso_forest.fit(X_train)

# Predecir sobre las comunidades reales
preds = iso_forest.predict(X_real)
anomaly_scores = iso_forest.decision_function(X_real)

comm_status = {}
report_rows = []

for idx, meta in enumerate(community_metadata):
    c_id = meta['comm_id']
    density = meta['density']
    avg_sim = meta['avg_sim']
    avg_iat_std = meta['avg_iat_std']

    # Banderas del modelo ML
    ml_is_bot = preds[idx] == -1
    raw_score = int(np.clip((0.5 - anomaly_scores[idx]) * 100, 0, 100))

    # SALVAGUARDAS (Regla determinista de alta seguridad):
    # Si la densidad y la similitud semántica son máximas, FORZAR detección de botnet
    safeguard_triggered = (density >= 0.8 and avg_sim >= 0.8) or (avg_iat_std <= 1.0 and avg_sim >= 0.8)

    is_bot = ml_is_bot or safeguard_triggered
    final_score = max(raw_score, 95) if safeguard_triggered else raw_score

    comm_status[c_id] = {
        'is_bot': is_bot,
        'score': final_score,
        'sim': round(avg_sim, 4),
        'density': round(density, 3),
        'iat_std': round(avg_iat_std, 2)
    }

    report_rows.append({
        'Comunidad': c_id,
        'Usuarios': len(meta['users']),
        'Densidad Grafo': round(density, 3),
        'Simil. Semántica': round(avg_sim, 4),
        'Varianza Temp. (s)': round(avg_iat_std, 2),
        'Risk Score ML': final_score,
        'Clasificación ML': 'BOTNET (Anomalía)' if is_bot else 'HUMANO (Normal)'
    })

print("\n--- INFORME MULTI-CAPA CORREGIDO (HÍBRIDO ML + SAFEGUARDS) ---")
print(pd.DataFrame(report_rows).to_string(index=False))

# ==========================================
# 5. GENERACIÓN DEL GRAFO INTERACTIVO (PyVis)
# ==========================================
net = Network(height='750px', width='100%', bgcolor='#222222', font_color='white', notebook=False)
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=100)

for node in G.nodes():
    c_id = partition[node]
    status = comm_status[c_id]

    if status['is_bot']:
        color = '#ff4d4d'  # Rojo para Bots
        label_class = 'BOTNET (Anomalía)'
        size = 25
    else:
        color = '#4dda53'  # Verde para Humanos
        label_class = 'HUMANO (Normal)'
        size = 15

    hover_title = f"""
    <b>Usuario:</b> {node}<br>
    <b>Comunidad:</b> {c_id}<br>
    <b>Clasificación:</b> {label_class}<br>
    <b>Risk Score:</b> {status['score']}/100<br>
    <hr>
    <i>Métricas de Comunidad:</i><br>
    • Densidad Grafo: {status['density']}<br>
    • Similitud Semántica: {status['sim']}<br>
    • Varianza Temp (IAT): {status['iat_std']}s
    """

    net.add_node(node, label=node, title=hover_title, color=color, size=size, shape='dot')

for src, dst in G.edges():
    net.add_edge(src, dst, color='#888888', width=1.5)

output_file = "grafo_botnets_interactivo.html"
net.write_html(output_file)
print(f"\n✅ Grafo interactivo exportado exitosamente a: {output_file}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


--- INFORME MULTI-CAPA CORREGIDO (HÍBRIDO ML + SAFEGUARDS) ---
 Comunidad  Usuarios  Densidad Grafo  Simil. Semántica  Varianza Temp. (s)  Risk Score ML  Clasificación ML
         0         3           0.667            0.0993               200.0             51 BOTNET (Anomalía)
         1         2           1.000            1.0000                 0.0             95 BOTNET (Anomalía)


NameError: name 'Network' is not defined

In [ ]:
# Set de semillas para reproducibilidad
np.random.seed(42)

# ==========================================
# 1. GENERACIÓN DE GRAFO (MÓDULO DE RED)
# ==========================================
G = nx.erdos_renyi_graph(n=80, p=0.03, seed=42)  # Red humana dispersa
bot_net = nx.complete_graph(20)                  # Botnet interconectada
G = nx.disjoint_union(G, bot_net)
G.add_edges_from([(80, 5), (81, 12), (82, 34)])

# Detectar comunidades
partition = community_louvain.best_partition(G)
nx.set_node_attributes(G, partition, 'community')

# ==========================================
# 2. GENERACIÓN DE EMBEDDINGS (SEMÁNTICA)
# ==========================================
embeddings = np.random.randn(100, 384)

# Inyección semántica en la botnet (nodos 80 a 99)
bot_nodes = list(range(80, 100))
seed_vector = np.random.randn(1, 384)
for node in bot_nodes:
    embeddings[node] = seed_vector + np.random.normal(0, 0.05, 384)

sim_matrix = cosine_similarity(embeddings)

# ==========================================
# 3. GENERACIÓN DE MARCAS TEMPORALES (TELEMETRÍA)
# ==========================================
# Generamos marcas de tiempo (timestamps en segundos) para cada usuario
user_timestamps = {}

for node in range(100):
    if node in bot_nodes:
        # BOTS: Publicaciones coordinadas casi en ráfaga (intervalos muy uniformes)
        # Ráfaga sincronizada con muy poca varianza (p. ej. cada 2 segundos ± 0.1s)
        base_time = 1700000000
        timestamps = base_time + np.cumsum(np.random.normal(loc=2.0, scale=0.1, size=10))
    else:
        # HUMANOS: Intervalos heterogéneos y dispersos (Poisson variable)
        base_time = 1700000000 + np.random.randint(0, 86400)
        timestamps = base_time + np.cumsum(np.random.exponential(scale=300.0, size=10))

    user_timestamps[node] = timestamps

# ==========================================
# 4. EVALUACIÓN Y MOTOR DE DECISIÓN MULTI-CAPA
# ==========================================
results = []
for comm_id in set(partition.values()):
    nodes_in_comm = [node for node, c in partition.items() if c == comm_id]

    if len(nodes_in_comm) > 1:
        # A) Métrica Semántica
        sub_matrix = sim_matrix[np.ix_(nodes_in_comm, nodes_in_comm)]
        mask = ~np.eye(sub_matrix.shape[0], dtype=bool)
        avg_sim = sub_matrix[mask].mean()

        # B) Métrica de Grafo
        sub_graph = G.subgraph(nodes_in_comm)
        density = nx.density(sub_graph)

        # C) Métrica de Telemetría Temporal (IAT Std Dev)
        # Calculamos la desviación estándar promedio del intervalo entre publicaciones
        comm_iat_std = []
        for n in nodes_in_comm:
            ts = user_timestamps[n]
            iats = np.diff(ts)  # Intervalos inter-arribo
            comm_iat_std.append(np.std(iats))

        avg_iat_std = np.mean(comm_iat_std)

        # MOTOR DE REGLAS / RISK SCORE (0 a 100)
        risk_score = 0
        if density > 0.5: risk_score += 35
        if avg_sim > 0.7: risk_score += 40
        if avg_iat_std < 1.0: risk_score += 25  # Varianza temporal sospechosamente baja

        sospecha = 'ALTA' if risk_score >= 60 else ('MEDIA' if risk_score >= 30 else 'BAJA')

        results.append({
            'Comunidad': comm_id,
            'Nodos': len(nodes_in_comm),
            'Densidad Grafo': round(density, 3),
            'Simil. Semántica': round(avg_sim, 4),
            'Varianza Temp. (s)': round(avg_iat_std, 2),
            'Risk Score': risk_score,
            'Sospecha': sospecha
        })

df = pd.DataFrame(results)
print("--- ANÁLISIS INTEGRADO: GRAFO + SEMÁNTICA + TELEMETRÍA TEMPORAL ---")
print(df.to_string(index=False))

In [ ]:
!pip install fastapi uvicorn pydantic

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import pandas as pd
import numpy as np
import networkx as nx
from community import community_louvain
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import IsolationForest

app = FastAPI(
    title="Botnet & Synthetic Content Detection API",
    description="Motor multi-capa de detección de bots mediante Grafos, Embeddings Semánticos y Telemetría Temporal.",
    version="1.0.0"
)

# Cargar el modelo de NLP al iniciar la API
print("Cargando modelo SentenceTransformer en memoria...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Estructura del JSON de entrada
class PostData(BaseModel):
    user_id: str
    text: str
    created_at: str  # Formato: "YYYY-MM-DD HH:MM:SS"
    mentioned_user: Optional[str] = None

class DetectionRequest(BaseModel):
    posts: List[PostData]

@app.post("/analyze")
def analyze_network(payload: DetectionRequest):
    if len(payload.posts) < 2:
        raise HTTPException(status_code=400, detail="Se requieren al menos 2 publicaciones para construir el grafo de interacción.")

    # 1. Convertir JSON a DataFrame
    data = [p.dict() for p in payload.posts]
    df = pd.DataFrame(data)
    df['timestamp'] = pd.to_datetime(df['created_at']).astype('int64') // 10**9

    # 2. Construcción del Grafo
    G = nx.Graph()
    for _, row in df.iterrows():
        u_src = str(row['user_id'])
        u_dst = str(row['mentioned_user']) if pd.notna(row.get('mentioned_user')) else None
        G.add_node(u_src)
        if u_dst and u_dst != 'None' and u_dst != u_src:
            G.add_edge(u_src, u_dst)

    partition = community_louvain.best_partition(G)

    # 3. Embeddings y Matriz de Similitud
    text_embeddings = model.encode(df['text'].tolist(), show_progress_bar=False)
    df['embedding'] = list(text_embeddings)

    user_embeddings = {}
    user_timestamps = {}
    for user, group in df.groupby('user_id'):
        u_str = str(user)
        user_embeddings[u_str] = np.mean(np.array(group['embedding'].tolist()), axis=0)
        user_timestamps[u_str] = group['timestamp'].sort_values().tolist()

    embedding_dim = text_embeddings.shape[1]
    all_users = list(partition.keys())
    for u in all_users:
        if u not in user_embeddings:
            user_embeddings[u] = np.zeros(embedding_dim)
            user_timestamps[u] = []

    user_matrix = np.array([user_embeddings[u] for u in all_users])
    user_to_idx = {user: idx for idx, user in enumerate(all_users)}
    sim_matrix = cosine_similarity(user_matrix)

    # 4. Extracción de Métricas por Comunidad
    features_list = []
    community_metadata = []

    for comm_id in set(partition.values()):
        users_in_comm = [user for user, c in partition.items() if c == comm_id]

        indices = [user_to_idx[u] for u in users_in_comm]
        sub_matrix = sim_matrix[np.ix_(indices, indices)]
        mask = ~np.eye(sub_matrix.shape[0], dtype=bool)
        avg_sim = sub_matrix[mask].mean() if mask.any() else 0.0

        sub_graph = G.subgraph(users_in_comm)
        density = nx.density(sub_graph)

        comm_iat_std = []
        for u in users_in_comm:
            ts = user_timestamps[u]
            if len(ts) > 1:
                iats = np.diff(ts)
                comm_iat_std.append(np.std(iats))
            else:
                comm_iat_std.append(300.0)
        avg_iat_std = np.mean(comm_iat_std)

        features_list.append([density, avg_sim, avg_iat_std])
        community_metadata.append({
            'comm_id': comm_id,
            'users': users_in_comm,
            'density': float(density),
            'avg_sim': float(avg_sim),
            'avg_iat_std': float(avg_iat_std)
        })

    X_real = np.array(features_list)

    # 5. Modelo ML + Safeguards
    baseline_human_data = np.array([
        [0.10, 0.01, 250.0],
        [0.15, -0.02, 180.0],
        [0.08, 0.05, 320.0],
        [0.20, 0.00, 150.0]
    ])
    X_train = np.vstack([baseline_human_data, X_real])

    iso_forest = IsolationForest(contamination=0.15, random_state=42)
    iso_forest.fit(X_train)

    preds = iso_forest.predict(X_real)
    anomaly_scores = iso_forest.decision_function(X_real)

    results = []
    for idx, meta in enumerate(community_metadata):
        density = meta['density']
        avg_sim = meta['avg_sim']
        avg_iat_std = meta['avg_iat_std']

        ml_is_bot = preds[idx] == -1
        raw_score = int(np.clip((0.5 - anomaly_scores[idx]) * 100, 0, 100))

        safeguard_triggered = (density >= 0.8 and avg_sim >= 0.8) or (avg_iat_std <= 1.0 and avg_sim >= 0.8)

        is_bot = (ml_is_bot and raw_score >= 60) or safeguard_triggered
        final_score = max(raw_score, 95) if safeguard_triggered else raw_score

        results.append({
            "community_id": meta['comm_id'],
            "users_count": len(meta['users']),
            "users_list": meta['users'],
            "metrics": {
                "graph_density": round(density, 3),
                "semantic_similarity": round(avg_sim, 4),
                "time_variance_seconds": round(avg_iat_std, 2)
            },
            "risk_score": final_score,
            "classification": "BOTNET" if is_bot else "HUMAN"
        })

    return {
        "status": "success",
        "communities_analyzed": len(results),
        "results": results
    }